In [ ]:
#MRDS
# Here we answer to the second reviwer cnn for MINST
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 123
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ----------------------------
# Load MNIST
# ----------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") / 255.0)[..., None]  # (N, 28, 28, 1)
x_test  = (x_test.astype("float32") / 255.0)[..., None]

num_classes = 10
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_oh  = tf.keras.utils.to_categorical(y_test, num_classes)

N_train = x_train.shape[0]
N_test  = x_test.shape[0]

# ----------------------------
#We create small tractable model
# Model factory (small CNN)
# ----------------------------
def make_small_cnn(seed: int) -> tf.keras.Model:
    # Each candidate differs via initialization seed (and later training subset),
    # producing a diverse finite hypothesis set.
    tf.keras.utils.set_random_seed(seed)
    inputs = tf.keras.Input(shape=(28, 28, 1))
    x = tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same")(inputs)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model
# ----------------------------
# Pretrain a finite set of candidate models
# ----------------------------
def pretrain_candidates(
    R: int = 25,
    pretrain_subset: int = 10000,
    pretrain_epochs: int = 1,
    batch_size: int = 128
):
    candidates = []
    # fixed pool for pretraining subset selection (still random)
    idx_pool = np.random.permutation(N_train)

    for r in range(R):
        # Each candidate gets a different seed and different subset slice
        seed_r = SEED + 1000 + r
        model = make_small_cnn(seed_r)

        # Different subset per candidate (shifted window over a permuted pool)
        start = (r * pretrain_subset) % (N_train - pretrain_subset)
        idx = idx_pool[start:start + pretrain_subset]

        model.fit(
            x_train[idx], y_train_oh[idx],
            epochs=pretrain_epochs,
            batch_size=batch_size,
            verbose=0
        )
        candidates.append(model)

        # Optional: quick sanity print
        if (r + 1) % max(1, R // 5) == 0:
            loss, acc = model.evaluate(x_test, y_test_oh, verbose=0)
            print(f"[pretrain] candidate {r+1:>3}/{R} | test acc ~ {acc:.4f}")

    return candidates

# ----------------------------
# Utility: classification error (0-1 loss)
# ----------------------------
@tf.function
def batch_error(model: tf.keras.Model, x, y_true_int):
    # returns mean 0-1 error on batch
    probs = model(x, training=False)
    y_pred = tf.argmax(probs, axis=1, output_type=tf.int32)
    err = tf.cast(tf.not_equal(y_pred, y_true_int), tf.float32)
    return tf.reduce_mean(err)

def eval_error(model, x, y_true_int, batch_size=512):
    # Evaluate error in batches to be safe on memory
    n = x.shape[0]
    errs = []
    for i in range(0, n, batch_size):
        xb = x[i:i+batch_size]
        yb = y_true_int[i:i+batch_size]
        errs.append(float(batch_error(model, xb, yb).numpy()))
    return float(np.mean(errs))

# ----------------------------
#Here
# Main experiment: discrete selection with q(W|S)
# ----------------------------
def run_mnist_discrete_selection_experiment(
    candidates,
    m: int = 256,
    n_datasets: int = 300,
    alpha: float = 0.7,
    batch_eval_S: int = 512
):
    """
    For each dataset S of size m:
      - pick r* = argmin_r empirical error on S
      - define q_r(S) = (1-alpha)/R + alpha * 1[r=r*]
      - sample W ~ q(.|S)
      - compute train error (on S) and test error (MNIST test)
    Also compute:
      - H(W|S) averaged over S via entropy(q(.|S))
      - H(W) from empirical marginal p(W)=E_S[q(.|S)]
      - I(S;W)=H(W)-H(W|S) in nats
      - bound = sqrt(2 I / m)
    """
    R = len(candidates)
    q_list = np.zeros((n_datasets, R), dtype=np.float64)
    chosen = np.zeros(n_datasets, dtype=np.int32)
    train_errs = np.zeros(n_datasets, dtype=np.float64)
    test_errs  = np.zeros(n_datasets, dtype=np.float64)

    y_test_int = y_test.astype(np.int32)

    for t in range(n_datasets):
        # Sample S (without replacement)
        idx = np.random.choice(N_train, size=m, replace=False)
        xS = x_train[idx]
        yS_int = y_train[idx].astype(np.int32)

        # Compute empirical errors of each candidate on S -
        emp_errs = np.zeros(R, dtype=np.float64)
        for r, model in enumerate(candidates):
            emp_errs[r] = eval_error(model, xS, yS_int, batch_size=batch_eval_S)

        r_star = int(np.argmin(emp_errs))

        # Build q(.|S)
        q = np.full(R, (1.0 - alpha) / R, dtype=np.float64)
        q[r_star] += alpha
        q_list[t] = q

        # Sample W ~ q
        w = int(np.random.choice(R, p=q))
        chosen[t] = w

        # Train error on S for chosen model
        train_errs[t] = eval_error(candidates[w], xS, yS_int, batch_size=batch_eval_S)

        # Test error for chosen model
        test_errs[t] = eval_error(candidates[w], x_test, y_test_int, batch_size=1024)

        if (t + 1) % max(1, n_datasets // 5) == 0:
            print(f"[trial] {t+1:>4}/{n_datasets} | "
                  f"mean gap so far = {np.mean(test_errs[:t+1] - train_errs[:t+1]):.4f}")

    # Generalization gap (expected R - R_S)
    gaps = test_errs - train_errs
    mean_gap = float(np.mean(gaps))
    mean_abs_gap = float(np.mean(np.abs(gaps)))

    # Compute entropies
    # H(W|S) ≈ E_S[ -sum_r q_r(S) log q_r(S) ]
    eps = 1e-12
    H_W_given_S = float(np.mean(-np.sum(q_list * np.log(q_list + eps), axis=1)))

    # p(W) ≈ average of q(.|S)
    pW = np.mean(q_list, axis=0)
    H_W = float(-np.sum(pW * np.log(pW + eps)))

    I_S_W = float(H_W - H_W_given_S)  # in nats

    # Bound term (nats version)
    bound = float(np.sqrt(2.0 * I_S_W / m))

    results = {
        "m": m,
        "n_datasets": n_datasets,
        "R": R,
        "alpha": alpha,
        "mean_train_error": float(np.mean(train_errs)),
        "mean_test_error": float(np.mean(test_errs)),
        "mean_gap": mean_gap,
        "mean_abs_gap": mean_abs_gap,
        "H_W": H_W,
        "H_W_given_S": H_W_given_S,
        "I_S_W": I_S_W,
        "bound_sqrt_2I_over_m": bound,
        "pW": pW,
        "gaps": gaps,
    }
    return results

# ----------------------------
# Run everything
# ----------------------------
if __name__ == "__main__":
    # 1) Pretrain finite candidate set
    R = 25                 # number of candidate neural nets
    pretrain_subset = 10000
    pretrain_epochs = 1    # keep it cheap; increase to 2-3 if you want stronger models

    candidates = pretrain_candidates(
        R=R,
        pretrain_subset=pretrain_subset,
        pretrain_epochs=pretrain_epochs,
        batch_size=128
    )

    # 2) Run the discrete-selection experiment
    m = 256
    n_datasets = 300
    alpha = 0.7

    results = run_mnist_discrete_selection_experiment(
        candidates=candidates,
        m=m,
        n_datasets=n_datasets,
        alpha=alpha
    )

    # 3) Print summary
    print("\n=== Summary ===")
    print(f"R = {results['R']}, m = {results['m']}, n_datasets = {results['n_datasets']}, alpha = {results['alpha']}")
    print(f"Mean train error: {results['mean_train_error']:.4f}")
    print(f"Mean test  error: {results['mean_test_error']:.4f}")
    print(f"Mean gap (test - train): {results['mean_gap']:.4f}")
    print(f"Mean |gap|: {results['mean_abs_gap']:.4f}")
    print(f"H(W)         = {results['H_W']:.4f} nats")
    print(f"H(W|S)       = {results['H_W_given_S']:.4f} nats")
    print(f"I(S;W)       = {results['I_S_W']:.4f} nats")
    print(f"sqrt(2 I / m)= {results['bound_sqrt_2I_over_m']:.4f}")

    # 4) Optional: plot gaps distribution + bound
    gaps = results["gaps"]
    plt.figure()
    plt.hist(gaps, bins=30, density=True)
    plt.axvline(results["mean_gap"], linestyle="--")
    plt.axvline(+results["bound_sqrt_2I_over_m"], linestyle=":")
    plt.axvline(-results["bound_sqrt_2I_over_m"], linestyle=":")
    plt.title("Generalization gap distribution and MI bound proxy")
    plt.xlabel("test error - train error")
    plt.ylabel("density")
    plt.show()


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[pretrain] candidate   5/25 | test acc ~ 0.9133
[pretrain] candidate  10/25 | test acc ~ 0.9142
[pretrain] candidate  15/25 | test acc ~ 0.9137
[pretrain] candidate  20/25 | test acc ~ 0.8906
